# Welcome to my Beautiful Dark Twisted Fantasy

# Make DataFrame

In [1]:
import Predictor
import pandas as pd
import hvplot.pandas
import panel as pn
import sklearn
import sklearn.model_selection
import sklearn.metrics
import numpy as np
import xarray as xr
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

pn.extension()

Data = Predictor.Clean(filePath="C:\\Users\\sophi\\OneDrive\\Documents\\BigAssBudgetData.xlsx")
df = Data.copy()
df['DescCodes'], uniques = df['Description'].factorize()
df['CatCodes'], uniques = df['Category'].factorize()


# Classify and Plot

In [2]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.inspection import DecisionBoundaryDisplay
data: pd.DataFrame = df[['DescCodes','Amount']]
target = df['CatCodes']
X,y = data, target
allData = pd.DataFrame(X,y)
X_Train, X_Test, y_train, y_test = sklearn.model_selection.train_test_split(X,y, test_size=0.1, random_state=42)
clf = make_pipeline(StandardScaler(),DecisionTreeClassifier(max_depth=5, random_state=42))
clf.fit(X_Train, y_train)
score = clf.score(X_Test, y_test)

In [3]:
import hvplot.xarray  # noqa
import numpy as np
import xarray as xr
from sklearn.svm import SVC
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

model = SVC(kernel='rbf').fit(X_Train,y_train)
linModel = LinearRegression()
treeModel = DecisionTreeRegressor()

model.fit(X_Train, y_train)
linModel.fit(X_Train, y_train)
treeModel.fit(X_Train, y_train)

xMin, xMax = X_Train['DescCodes'].min() - 1, X_Train['DescCodes'].max() + 1
yMin, yMax = X_Train['Amount'].min() - 10, X_Train['Amount'].max() + 10

xArray, yArray = np.arange(xMin, xMax), np.arange(yMin, yMax)
X, Y =  np.meshgrid(xArray, yArray)
points = np.c_[X.ravel(), Y.ravel()]

Z = model.predict(points).reshape(X.shape)
Z2 = linModel.predict(points).reshape(X.shape)
Z3 = treeModel.predict(points).reshape(X.shape)

ds = xr.DataArray(Z, coords=[('y',yArray),('x', xArray)], name='z').to_dataset()
ds2 = xr.DataArray(Z2, coords=[('Amount', yArray),('DescCodes', xArray)], name='z2')
ds3 = xr.DataArray(Z3, coords=[('Amount', yArray),('DescCodes', xArray)], name='z3')

ds.hvplot.contour(x='x', y='y', z='z')
ds2.hvplot.contour(x='DescCodes', y='Amount', z='z2')

boundary = ds3.hvplot.contourf(x='DescCodes', y='Amount', z='z3')
data = allData.hvplot.scatter(x='DescCodes', y='Amount', by='CatCodes' )

(boundary * data).opts(title='hello')

c:\Users\sophi\Git_Root\Python\BudgetApp\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but SVC was fitted with feature names
  warnings.warn(
c:\Users\sophi\Git_Root\Python\BudgetApp\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
c:\Users\sophi\Git_Root\Python\BudgetApp\venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but DecisionTreeRegressor was fitted with feature names
  warnings.warn(


:Overlay
   .Polygons.I  :Polygons   [DescCodes,Amount]   (z3)
   .NdOverlay.I :NdOverlay   [CatCodes]
      :Scatter   [DescCodes]   (Amount)

In [ ]:

# gridX, gridY = np.meshgrid(np.arange(xMin, xMax, 10), np.arange(yMin, yMax, 2))

# gridPoints = np.c_[gridX.ravel(), gridY.ravel()]
# Z = model.predict(gridPoints).reshape(gridX.shape)
# GridFrame= pd.DataFrame({'DescCodes' : gridX.ravel(), 'Amount':gridY.ravel(), 'Category': Z.ravel()}) 
# data = X_Test

# (GridFrame * data).hvplot.opts()

